<div style="text-align: center;">

  <img src="img/LogoUesc.png" alt="UESC" style="width: 60px; display: block; margin: 0 auto 5px auto;">

  <div style="font-size: 20px;"><strong>UNIVERSIDADE ESTADUAL DE SANTA CRUZ - UESC</strong></div>
  <div style="font-size: 18px;">DEPARTAMENTO DE ENGENHARIAS E COMPUTAÇÃO - DEC</div>
  <div style="font-size: 18px;">Grupo de Pesquisas em Simulações e Controle de Processos</div>

  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

  <div style="font-size: 22px; font-weight: bold; margin-bottom: 10px;">PROGRAMAÇÃO EM PYTHON PARA AS ENGENHARIAS</div>

  <div style="font-size: 18px; margin-top: 10px;"><strong>Tema</strong>: Construindo funções confiáveis para aplicações de Engenharia</div>
  <div style="font-size: 18px;"><strong>Prof.  Dr.  E. R. Edwards</strong></div>

  <div style="font-size: 16px; margin-top: 15px;">Ilhéus - BA, Abril de 2027</div>
  
  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

</div>

**__Sumário__**

- [1. Por que precisamos de funções confiáveis?](#secao-1)
- [2. Identificando entradas fisicamente inválidas](#secao-2)
- [3. Validando dados com if e raise](#secao-3)
- [4. Tratando erros com try e except](#secao-4)
- [5. Documentando funções com docstrings](#secao-5)
- [6. Testando funções com assert e math.isclose](#secao-6)
- [7. Instrumentação: sinais de transmissores de 4–20 mA](#secao-7)
- [8. Controle de Processos: verificando erros e limites](#secao-8)
- [9. Mecânica dos Fluidos: verificando modelos de escoamento](#secao-9)
- [10. Transferência de Calor: sinais, unidades e resultados](#secao-10)
- [11. Integração, exercícios e conclusão](#secao-11)

<div id="secao-1" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  1. Por que precisamos de funções confiáveis?
</div>

Nas aulas anteriores, aprendemos a criar funções com `def`, devolver resultados com `return` e combinar funções para resolver problemas de Engenharia. Entretanto, um código que funciona para um conjunto de dados pode produzir resultados sem significado físico quando recebe outros valores.

Considere a velocidade média de um fluido em uma tubulação circular: $v=Q/A$, com $A=\pi D^2/4$. Se alguém informar `D = 0`, ocorrerá uma divisão por zero. Se informar `D = -0.05`, a expressão $D^2$ ainda produzirá uma área positiva, embora um diâmetro negativo não tenha significado físico.

**Uma função confiável não apenas calcula: ela também explicita suas hipóteses, verifica entradas e permite testar seus resultados.**

In [1]:
import math

def velocidade_simples(Q, D):
    area = math.pi * D**2 / 4
    return Q / area

print(f"Velocidade = {velocidade_simples(0.002, 0.05):.3f} m/s")

Velocidade = 1.019 m/s


<div id="secao-2" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  2. Identificando entradas fisicamente inválidas
</div>

A primeira pergunta não é “como evitar um erro de Python?”, mas **“quais valores fazem sentido para este modelo?”**. Em um escoamento convencional, o diâmetro interno deve ser maior que zero; a vazão volumétrica pode ser positiva, nula ou negativa se o sinal indicar o sentido do escoamento.

A validação depende da **definição física** das variáveis. Em transferência de calor, por exemplo, uma diferença de temperatura negativa pode ser perfeitamente válida: ela indica a inversão do sentido de transferência segundo a convenção adotada.

Devemos distinguir: (a) entradas impossíveis; (b) entradas fora do domínio de validade do modelo; e (c) resultados fisicamente possíveis, mesmo quando negativos.

<div id="secao-3" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  3. Validando dados com if e raise
</div>

O comando `if` permite examinar uma condição. Com `raise ValueError(...)`, interrompemos o cálculo e informamos por que a entrada não é aceitável. Aqui, a condição `D <= 0` é verificada **antes** da divisão.

In [2]:
import math

def velocidade_media(Q, D):
    if D <= 0:
        raise ValueError("O diâmetro deve ser maior que zero.")
    area = math.pi * D**2 / 4
    return Q / area

v = velocidade_media(0.002, 0.05)
print(f"Velocidade média = {v:.3f} m/s")

Velocidade média = 1.019 m/s


**Verificação manual.** Para $D=0{,}05$ m e $Q=0{,}002$ m³/s, a área é aproximadamente $0{,}0019635$ m² e a velocidade é aproximadamente $1{,}019$ m/s. O valor calculado deve concordar com essa estimativa.

**Experimente:** substitua o diâmetro por zero. Observe a mensagem do erro e identifique a linha que impede o cálculo inválido.

<div id="secao-4" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  4. Tratando erros com try e except
</div>

Uma função deve sinalizar o problema; o programa que a utiliza decide como responder. `try` tenta executar uma operação e `except ValueError` trata especificamente um erro de valor. Não devemos esconder todos os erros com um `except` genérico.

In [3]:
diametros = [0.05, 0.0, 0.08]
Q = 0.002

for D in diametros:
    try:
        v = velocidade_media(Q, D)
        print(f"D = {D:.3f} m -> v = {v:.3f} m/s")
    except ValueError as erro:
        print(f"D = {D:.3f} m -> entrada rejeitada: {erro}")

D = 0.050 m -> v = 1.019 m/s
D = 0.000 m -> entrada rejeitada: O diâmetro deve ser maior que zero.
D = 0.080 m -> v = 0.398 m/s


Observe que o programa continua processando os demais diâmetros. **Não substituímos automaticamente o dado inválido por zero**: isso poderia contaminar os resultados e esconder um problema experimental.

<div id="secao-5" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  5. Documentando funções com docstrings
</div>

Uma *docstring* é um texto colocado logo após a linha `def`. Ela informa o propósito da função, os parâmetros, as unidades, o resultado e as condições em que um erro será gerado. Isso é especialmente importante quando colegas ou estudantes reutilizam o código.

In [4]:
def vazao_massica(rho, Q):
    """Calcula a vazão mássica de um fluido.

    Parâmetros
    ----------
    rho : float
        Massa específica em kg/m³ (deve ser positiva).
    Q : float
        Vazão volumétrica em m³/s; sinal indica direção.

    Retorna
    -------
    float
        Vazão mássica em kg/s.

    Lança
    -----
    ValueError
        Se a massa específica não for positiva.
    """
    if rho <= 0:
        raise ValueError("A massa específica deve ser positiva.")
    return rho * Q

print(vazao_massica(1000, 0.003))
print(vazao_massica.__doc__.splitlines()[0])

3.0
Calcula a vazão mássica de um fluido.


<div id="secao-6" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  6. Testando funções com assert e math.isclose
</div>

Testar é comparar um resultado computacional com um valor conhecido ou com uma propriedade esperada. `assert` permite registrar uma condição que deve ser verdadeira. Para resultados de ponto flutuante, utilizamos `math.isclose()` em vez de exigir igualdade exata.

**Atenção:** `assert` é apropriado para testes de desenvolvimento; não substitui a validação de entradas em programas de produção, pois pode ser desativado na execução otimizada do Python.

In [5]:
import math

assert math.isclose(vazao_massica(1000, 0.003), 3.0, rel_tol=1e-12)
assert math.isclose(velocidade_media(0, 0.05), 0.0, abs_tol=1e-12)

try:
    velocidade_media(0.002, 0)
except ValueError:
    print("Teste de diâmetro inválido: aprovado")
else:
    raise AssertionError("Era esperado ValueError para D = 0")

print("Testes numéricos: aprovados")

Teste de diâmetro inválido: aprovado
Testes numéricos: aprovados


<div id="secao-7" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  7. Instrumentação: sinais de transmissores de 4–20 mA
</div>

Um transmissor ideal de temperatura configurado para medir de 0 a 200 °C produz, nominalmente, corrente entre 4 e 20 mA. Para uma relação linear, a temperatura pode ser estimada por:

$$T = T_{min} + \frac{I-4}{16}(T_{max}-T_{min})$$

Para $I=12$ mA, obtemos $T=100$ °C. Uma corrente fora de 4–20 mA não deve ser interpretada automaticamente como temperatura válida nessa faixa nominal; pode indicar uma condição fora da faixa ou sinalização de falha, conforme a configuração e o equipamento.

In [6]:
def temperatura_transmissor(I, Tmin=0, Tmax=200):
    """Converte corrente nominal de 4–20 mA em temperatura (°C)."""
    if Tmax <= Tmin:
        raise ValueError("A faixa de medição deve ser crescente.")
    if not 4 <= I <= 20:
        raise ValueError("Corrente fora da faixa nominal de 4–20 mA.")
    return Tmin + (I - 4) * (Tmax - Tmin) / 16

for corrente in [4, 12, 20]:
    print(f"{corrente:>2} mA -> {temperatura_transmissor(corrente):.1f} °C")

 4 mA -> 0.0 °C
12 mA -> 100.0 °C
20 mA -> 200.0 °C


**Interpretação:** a função não diagnostica a causa de uma corrente anormal. Ela apenas recusa a conversão nominal fora do domínio definido. O diagnóstico real depende do instrumento e de sua configuração.

<div id="secao-8" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  8. Controle de Processos: verificando erros e limites
</div>

Em Controle de Processos, o erro é $e=SP-PV$. Uma contribuição proporcional ideal pode ser calculada por $u_P=K_c e$. Se o ganho estiver em pontos percentuais por °C, o resultado será expresso em pontos percentuais.

Para $SP=80$ °C, $PV=76$ °C e $K_c=2$ pontos percentuais por °C, temos $e=4$ °C e $u_P=8$ pontos percentuais. Isso **não representa necessariamente a saída total do controlador**, que pode incluir viés e outras ações.

In [7]:
def erro_controle(SP, PV):
    return SP - PV

def contribuicao_proporcional(SP, PV, Kc):
    if Kc < 0:
        raise ValueError("Neste exemplo, Kc deve ser não negativo.")
    return Kc * erro_controle(SP, PV)

print("Erro:", erro_controle(80, 76), "°C")
print("Contribuição P:", contribuicao_proporcional(80, 76, 2), "pontos percentuais")

Erro: 4 °C
Contribuição P: 8 pontos percentuais


A restrição de ganho não negativo é uma **hipótese deste exemplo**, não uma regra universal de controle. O sinal do ganho depende da convenção de erro, da ação direta ou reversa e do processo.

<div id="secao-9" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  9. Mecânica dos Fluidos: verificando modelos de escoamento
</div>

O número de Reynolds para escoamento em uma tubulação circular pode ser calculado por:

$$Re = \frac{\rho vD}{\mu}$$

Com $\rho=1000$ kg/m³, $\mu=0{,}001$ Pa·s, $Q=0{,}002$ m³/s e $D=0{,}05$ m, esperamos aproximadamente $Re=50930$. Massa específica, viscosidade dinâmica e diâmetro devem ser positivos. A vazão será considerada não negativa neste exemplo.

In [8]:
def reynolds_tubo(rho, mu, Q, D):
    """Calcula Reynolds para vazão não negativa em tubo circular."""
    if rho <= 0 or mu <= 0 or D <= 0:
        raise ValueError("rho, mu e D devem ser positivos.")
    if Q < 0:
        raise ValueError("Neste modelo, Q deve ser não negativa.")
    v = velocidade_media(Q, D)
    return rho * v * D / mu

Re = reynolds_tubo(1000, 0.001, 0.002, 0.05)
print(f"Re = {Re:.1f}")
assert math.isclose(Re, 50929.5817894065, rel_tol=1e-10)

Re = 50929.6


**Interpretação:** para um tubo circular e condições usuais de escoamento interno, esse Reynolds é compatível com regime turbulento. O cálculo do número de Reynolds, isoladamente, não valida todas as hipóteses do modelo hidráulico.

<div id="secao-10" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  10. Transferência de Calor: sinais, unidades e resultados
</div>

Para condução unidimensional estacionária através de uma parede plana, com condutividade térmica constante, a taxa de calor pode ser expressa por:

$$\dot Q = \frac{kA(T_1-T_2)}{L}$$

Com $k=0{,}8$ W/(m·K), $A=2$ m², $L=0{,}1$ m, $T_1=80$ °C e $T_2=30$ °C, resulta $\dot Q=800$ W. Se invertermos as temperaturas, o sinal muda: isso não é um erro de entrada.

In [9]:
def taxa_calor_parede(k, A, L, T1, T2):
    """Taxa de calor (W), positiva de 1 para 2."""
    if k <= 0 or A <= 0 or L <= 0:
        raise ValueError("k, A e L devem ser positivos.")
    return k * A * (T1 - T2) / L

q12 = taxa_calor_parede(0.8, 2, 0.1, 80, 30)
q21 = taxa_calor_parede(0.8, 2, 0.1, 30, 80)
print(f"Sentido 1 -> 2: {q12:.1f} W")
print(f"Sentido inverso: {q21:.1f} W")
assert math.isclose(q12, -q21)

Sentido 1 -> 2: 800.0 W
Sentido inverso: -800.0 W


<div id="secao-11" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  11. Integração, exercícios e conclusão
</div>

### Integrando as boas práticas

Ao construir um pequeno programa científico, procure manter uma sequência clara: **definir o fenômeno e as hipóteses → escrever a equação → implementar a função → validar entradas → testar resultados conhecidos → interpretar o resultado**.

No exemplo a seguir, diferentes correntes de um transmissor são processadas. Os valores inválidos são identificados sem impedir o processamento dos demais dados.

In [10]:
correntes = [4, 8, 12, 16, 20, 22]

for I in correntes:
    try:
        T = temperatura_transmissor(I)
        print(f"I = {I:>2} mA -> T = {T:>5.1f} °C")
    except ValueError as erro:
        print(f"I = {I:>2} mA -> {erro}")

I =  4 mA -> T =   0.0 °C
I =  8 mA -> T =  50.0 °C
I = 12 mA -> T = 100.0 °C
I = 16 mA -> T = 150.0 °C
I = 20 mA -> T = 200.0 °C
I = 22 mA -> Corrente fora da faixa nominal de 4–20 mA.


### Exercícios propostos

1. Modifique `temperatura_transmissor()` para uma faixa de -20 a 120 °C. Calcule as temperaturas correspondentes a 4, 12 e 20 mA e confira os resultados manualmente.
2. Acrescente uma verificação para impedir que a massa específica seja zero. Explique por que uma vazão negativa pode ser válida em um modelo com sentido de fluxo definido.
3. Escreva três testes para `taxa_calor_parede()`: um resultado conhecido, a inversão de temperaturas e uma espessura inválida.
4. Desenvolva uma função para a perda de carga em tubulação utilizando uma equação estudada em Mecânica dos Fluidos. Documente unidades e hipóteses, e identifique os parâmetros que precisam de validação.

### Conclusão

Uma função confiável torna explícitas as condições necessárias para o cálculo, informa entradas incompatíveis e permite verificar seus resultados. **O objetivo não é impedir toda situação inesperada, mas evitar que um número aparentemente correto seja confundido com uma resposta fisicamente válida.**

<div style="text-align: center; font-size: 12px; color: gray; margin-top: 40px;">
  Este notebook foi desenvolvido no âmbito do Grupo de Pesquisas em Modelagem Computacional da UESC.<br>
  Todos os direitos reservados © 2025
</div>